# CatVTON-FLUX LoRA Fine-Tuning

**Distillation Pipeline**: Trains CatVTON-FLUX LoRA using Gemini VTO outputs as ground truth.

## How it works
1. Downloads training triplets (person, garment, result) from Supabase
2. Generates agnostic masks using body segmentation
3. Fine-tunes CatVTON-FLUX LoRA on A100/H100
4. Uploads fine-tuned weights to Google Drive

**GPU**: A100 (40/80GB) or H100 required

**Estimated training time**:
- 200 pairs → ~2-3 hours on A100
- 500 pairs → ~5-6 hours on A100
- 1000 pairs → ~10-12 hours on A100

In [ ]:
#@title 1. Setup & Install Dependencies
import subprocess, os

# Check GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

# Clone CatVTON-FLUX repo
if not os.path.exists('/content/catvton-flux'):
    !git clone https://github.com/nftblackmagic/catvton-flux.git /content/catvton-flux

# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate bitsandbytes wandb
!pip install -q safetensors pillow requests tqdm
!pip install -q segment-anything-hq  # For mask generation

print("\n✅ Dependencies installed")

In [ ]:
#@title 2. Configuration

# ── Supabase config ──
SUPABASE_URL = "https://qfumhgipfhzubmorymbd.supabase.co"  #@param {type:"string"}
SUPABASE_SERVICE_KEY = ""  #@param {type:"string"}

# ── Training config ──
LORA_RANK = 16  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_TRAIN_STEPS = 5000  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
GRADIENT_ACCUMULATION = 4  #@param {type:"integer"}
IMAGE_HEIGHT = 768  #@param {type:"integer"}
IMAGE_WIDTH = 576  #@param {type:"integer"}
CHECKPOINT_STEPS = 500  #@param {type:"integer"}
VALIDATION_STEPS = 250  #@param {type:"integer"}

# ── Paths ──
DATA_DIR = "/content/training_data"
OUTPUT_DIR = "/content/catvton-flux-finetuned"
DRIVE_OUTPUT = "/content/drive/MyDrive/vto-models"  # Google Drive backup

# ── HuggingFace token (for downloading FLUX.1-dev) ──
HF_TOKEN = ""  #@param {type:"string"}

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"Config: LoRA rank={LORA_RANK}, lr={LEARNING_RATE}, steps={MAX_TRAIN_STEPS}")
print(f"Images: {IMAGE_WIDTH}x{IMAGE_HEIGHT}, batch={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION}")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

In [ ]:
#@title 3. Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"✅ Drive mounted. Checkpoints will save to: {DRIVE_OUTPUT}")

In [ ]:
#@title 4. Download Training Data from Supabase
import requests, json
from pathlib import Path
from tqdm import tqdm

# Create VITON-HD-compatible directory structure
for d in ['image', 'cloth', 'agnostic-mask', 'result']:
    os.makedirs(f"{DATA_DIR}/train/{d}", exist_ok=True)
    os.makedirs(f"{DATA_DIR}/test/{d}", exist_ok=True)

# Fetch training metadata from Supabase
headers = {
    "apikey": SUPABASE_SERVICE_KEY,
    "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}",
}

# Get all unused training data
res = requests.get(
    f"{SUPABASE_URL}/rest/v1/vto_training_data?used_in_training=eq.false&order=created_at.asc",
    headers=headers
)
records = res.json()
print(f"Found {len(records)} new training records")

if len(records) == 0:
    print("⚠️ No new training data available. Generate more via the kiosk!")
else:
    # Download images from Supabase storage
    train_pairs = []
    
    for i, rec in enumerate(tqdm(records, desc="Downloading")):
        rec_id = rec['id'][:8]
        
        def download_image(storage_path, local_path):
            url = f"{SUPABASE_URL}/storage/v1/object/vto-images/{storage_path}"
            r = requests.get(url, headers={"Authorization": f"Bearer {SUPABASE_SERVICE_KEY}"})
            if r.status_code == 200:
                with open(local_path, 'wb') as f:
                    f.write(r.content)
                return True
            return False
        
        # Download person, garment, result
        person_ok = download_image(rec['person_image_path'], f"{DATA_DIR}/train/image/{rec_id}.jpg")
        garment_ok = download_image(rec['garment_image_path'], f"{DATA_DIR}/train/cloth/{rec_id}.jpg")
        result_ok = download_image(rec['result_image_path'], f"{DATA_DIR}/train/result/{rec_id}.jpg")
        
        if person_ok and garment_ok and result_ok:
            train_pairs.append(f"{rec_id}.jpg {rec_id}.jpg")
        else:
            print(f"  ⚠️ Skipped {rec_id}: download failed")
    
    # Write train_pairs.txt
    with open(f"{DATA_DIR}/train_pairs.txt", 'w') as f:
        f.write('\n'.join(train_pairs))
    
    # Use 10% for validation
    split = max(1, len(train_pairs) // 10)
    with open(f"{DATA_DIR}/subtrain_1.txt", 'w') as f:
        f.write('\n'.join(train_pairs[-split:]))
    with open(f"{DATA_DIR}/subtest_1.txt", 'w') as f:
        f.write('\n'.join(train_pairs[-split:]))
    
    print(f"\n✅ Downloaded {len(train_pairs)} training pairs")
    print(f"   Train: {len(train_pairs) - split} | Validation: {split}")

In [ ]:
#@title 5. Generate Agnostic Masks
#
# CatVTON-FLUX needs inpainting masks that indicate WHERE to place the garment.
# We generate these automatically based on the category (upper_body/lower_body).
# For our distillation approach, we use the difference between person and result
# images to create precise masks.

from PIL import Image
import numpy as np
from pathlib import Path

def generate_mask_from_diff(person_path, result_path, output_path, threshold=30):
    """Generate inpainting mask by comparing person and Gemini result.
    
    The areas that changed = where the garment was placed = mask region.
    This is a clever trick: Gemini's output tells us exactly where the garment goes.
    """
    person = np.array(Image.open(person_path).convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT)))
    result = np.array(Image.open(result_path).convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT)))
    
    # Compute absolute difference
    diff = np.abs(person.astype(float) - result.astype(float)).mean(axis=2)
    
    # Threshold to get binary mask (changed regions)
    mask = (diff > threshold).astype(np.uint8) * 255
    
    # Dilate mask slightly to cover edges
    from PIL import ImageFilter
    mask_img = Image.fromarray(mask, mode='L')
    mask_img = mask_img.filter(ImageFilter.MaxFilter(7))  # Dilate
    mask_img = mask_img.filter(ImageFilter.GaussianBlur(3))  # Smooth edges
    
    # Re-threshold after blur
    mask_arr = np.array(mask_img)
    mask_arr = (mask_arr > 128).astype(np.uint8) * 255
    
    Image.fromarray(mask_arr, mode='L').save(output_path)
    return mask_arr.sum() > 0  # True if mask is non-empty


def generate_upper_body_mask(person_path, output_path):
    """Fallback: generate a simple upper-body rectangular mask."""
    img = Image.open(person_path).convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    mask = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.uint8)
    
    # Upper body region: roughly 15%-60% of height, 10%-90% of width
    y_start = int(IMAGE_HEIGHT * 0.15)
    y_end = int(IMAGE_HEIGHT * 0.60)
    x_start = int(IMAGE_WIDTH * 0.10)
    x_end = int(IMAGE_WIDTH * 0.90)
    mask[y_start:y_end, x_start:x_end] = 255
    
    Image.fromarray(mask, mode='L').save(output_path)


# Generate masks for all training images
pairs_file = f"{DATA_DIR}/train_pairs.txt"
if os.path.exists(pairs_file):
    with open(pairs_file) as f:
        pairs = [line.strip() for line in f if line.strip()]
    
    good_masks = 0
    for pair in tqdm(pairs, desc="Generating masks"):
        img_name = pair.split()[0]
        person_path = f"{DATA_DIR}/train/image/{img_name}"
        result_path = f"{DATA_DIR}/train/result/{img_name}"
        mask_path = f"{DATA_DIR}/train/agnostic-mask/{img_name.replace('.jpg', '_mask.png')}"
        
        if os.path.exists(result_path):
            # Use diff-based mask (best quality)
            if generate_mask_from_diff(person_path, result_path, mask_path):
                good_masks += 1
            else:
                # Fallback to rectangular mask
                generate_upper_body_mask(person_path, mask_path)
                good_masks += 1
        else:
            generate_upper_body_mask(person_path, mask_path)
            good_masks += 1
    
    print(f"\n✅ Generated {good_masks} masks")
else:
    print("⚠️ No training pairs found. Run step 4 first.")

In [ ]:
#@title 6. Prepare Custom Dataset Loader
#
# Our data format differs from standard VITON-HD:
# - We have Gemini results as ground truth (not real photos)
# - We use diff-based masks instead of segmentation masks
# This custom dataset bridges the gap.

CUSTOM_DATASET_CODE = '''
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

class GeminiDistillDataset(Dataset):
    """Dataset for distilling Gemini VTO outputs into CatVTON-FLUX.
    
    Each sample:
      - cloth: garment image
      - im_mask: person with masked garment region (inpainting input)
      - inpaint_mask: binary mask of garment region
      - GT_image: Gemini VTO result (ground truth for training)
    """
    
    def __init__(self, dataroot, pairs_file, height=768, width=576):
        self.dataroot = dataroot
        self.height = height
        self.width = width
        
        with open(os.path.join(dataroot, pairs_file)) as f:
            self.pairs = [line.strip().split() for line in f if line.strip()]
        
        self.transform = transforms.Compose([
            transforms.Resize((height, width)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # → [-1, 1]
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((height, width)),
            transforms.ToTensor(),
        ])
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        img_name, cloth_name = self.pairs[idx]
        
        # Load images
        person = Image.open(os.path.join(self.dataroot, "train/image", img_name)).convert("RGB")
        cloth = Image.open(os.path.join(self.dataroot, "train/cloth", cloth_name)).convert("RGB")
        result = Image.open(os.path.join(self.dataroot, "train/result", img_name)).convert("RGB")
        
        mask_name = img_name.replace(".jpg", "_mask.png")
        mask = Image.open(os.path.join(self.dataroot, "train/agnostic-mask", mask_name)).convert("L")
        
        # Transform
        person_t = self.transform(person)
        cloth_t = self.transform(cloth)
        result_t = self.transform(result)
        mask_t = self.mask_transform(mask)  # [0, 1]
        
        # Create masked person (person with garment region zeroed out)
        im_mask = person_t * (1 - mask_t)  # Keep non-mask regions
        
        return {
            "cloth_pure": cloth_t,
            "im_mask": im_mask,
            "inpaint_mask": mask_t,
            "GT_image": result_t,
            "im_name": img_name,
        }
'''

# Write custom dataset to the repo
with open('/content/catvton-flux/image_datasets/gemini_distill_dataset.py', 'w') as f:
    f.write(CUSTOM_DATASET_CODE)

print("✅ Custom dataset loader written")

In [ ]:
#@title 7. Start LoRA Fine-Tuning
#
# This trains the CatVTON-FLUX LoRA weights using Gemini outputs as ground truth.
# Uses flow matching loss (same as original FLUX training).

import subprocess

os.chdir('/content/catvton-flux')

# Build training command
cmd = f"""
accelerate launch --mixed_precision=bf16 train_flux_inpaint.py \
  --pretrained_model_name_or_path="black-forest-labs/FLUX.1-dev" \
  --pretrained_inpaint_model_name_or_path="xiaozaa/flux1-fill-dev-diffusers" \
  --output_dir="{OUTPUT_DIR}" \
  --mixed_precision="bf16" \
  --train_batch_size={BATCH_SIZE} \
  --gradient_accumulation_steps={GRADIENT_ACCUMULATION} \
  --learning_rate={LEARNING_RATE} \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --height={IMAGE_HEIGHT} \
  --width={IMAGE_WIDTH} \
  --max_sequence_length=512 \
  --dataroot="{DATA_DIR}" \
  --train_data_list="train_pairs.txt" \
  --train_verification_list="subtrain_1.txt" \
  --validation_data_list="subtest_1.txt" \
  --checkpointing_steps={CHECKPOINT_STEPS} \
  --validation_steps={VALIDATION_STEPS} \
  --train_lora \
  --lora_rank={LORA_RANK} \
  --gradient_checkpointing \
  --guidance_scale=1 \
  --weighting_scheme="logit_normal" \
  --seed=42
""".strip()

print("🚀 Starting LoRA fine-tuning...")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Steps: {MAX_TRAIN_STEPS}")
print(f"   LoRA rank: {LORA_RANK}")
print()

# Run training
process = subprocess.Popen(
    cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
print(f"\n{'✅ Training complete!' if process.returncode == 0 else '❌ Training failed!'}")

In [ ]:
#@title 8. Save Fine-Tuned LoRA to Google Drive
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
save_dir = f"{DRIVE_OUTPUT}/catvton-flux-lora-{timestamp}"
os.makedirs(save_dir, exist_ok=True)

# Copy the latest checkpoint
if os.path.exists(OUTPUT_DIR):
    # Find LoRA weights
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith('.safetensors') or f.endswith('.bin'):
            src = os.path.join(OUTPUT_DIR, f)
            shutil.copy2(src, save_dir)
            print(f"  Copied: {f} ({os.path.getsize(src)/1024/1024:.1f}MB)")
    
    # Also check checkpoint subdirs
    for d in sorted(os.listdir(OUTPUT_DIR)):
        cp_dir = os.path.join(OUTPUT_DIR, d)
        if os.path.isdir(cp_dir) and 'checkpoint' in d:
            cp_save = os.path.join(save_dir, d)
            shutil.copytree(cp_dir, cp_save)
            print(f"  Copied checkpoint: {d}")

# Save training config
config = {
    'lora_rank': LORA_RANK,
    'learning_rate': LEARNING_RATE,
    'max_train_steps': MAX_TRAIN_STEPS,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation': GRADIENT_ACCUMULATION,
    'image_size': f"{IMAGE_WIDTH}x{IMAGE_HEIGHT}",
    'training_samples': len(open(f"{DATA_DIR}/train_pairs.txt").readlines()) if os.path.exists(f"{DATA_DIR}/train_pairs.txt") else 0,
    'timestamp': timestamp,
}
import json
with open(os.path.join(save_dir, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Saved to Google Drive: {save_dir}")

In [ ]:
#@title 9. Upload LoRA Weights to Supabase Storage
#
# Upload the fine-tuned LoRA so the edge function can download it

import glob

# Find the latest LoRA weights file
lora_files = glob.glob(f"{OUTPUT_DIR}/**/*.safetensors", recursive=True)
if not lora_files:
    lora_files = glob.glob(f"{OUTPUT_DIR}/**/*.bin", recursive=True)

if lora_files:
    lora_path = sorted(lora_files)[-1]  # Latest
    lora_size = os.path.getsize(lora_path) / 1024 / 1024
    print(f"Uploading: {lora_path} ({lora_size:.1f}MB)")
    
    with open(lora_path, 'rb') as f:
        lora_bytes = f.read()
    
    # Upload to Supabase storage
    upload_path = f"models/catvton-flux-lora-latest.safetensors"
    r = requests.post(
        f"{SUPABASE_URL}/storage/v1/object/vto-images/{upload_path}",
        headers={
            "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}",
            "Content-Type": "application/octet-stream",
            "x-upsert": "true",
        },
        data=lora_bytes
    )
    
    if r.status_code in (200, 201):
        print(f"✅ Uploaded to: vto-images/{upload_path}")
    else:
        print(f"❌ Upload failed: {r.status_code} {r.text[:200]}")
else:
    print("⚠️ No LoRA weights found. Training may not have completed.")

In [ ]:
#@title 10. Mark Training Data as Used
#
# Update Supabase to mark these records as used in training,
# so the next training run only processes new data.

batch_name = f"batch-{timestamp}"

# Get IDs of records we trained on
res = requests.get(
    f"{SUPABASE_URL}/rest/v1/vto_training_data?used_in_training=eq.false&select=id",
    headers=headers
)
ids = [r['id'] for r in res.json()]

if ids:
    # Batch update
    for rec_id in ids:
        requests.patch(
            f"{SUPABASE_URL}/rest/v1/vto_training_data?id=eq.{rec_id}",
            headers={**headers, "Content-Type": "application/json", "Prefer": "return=minimal"},
            json={"used_in_training": True, "training_batch": batch_name}
        )
    print(f"✅ Marked {len(ids)} records as used (batch: {batch_name})")
else:
    print("No records to update")

In [ ]:
#@title 11. Quick Inference Test
#
# Test the fine-tuned model on a sample image

import torch
from diffusers import FluxFillPipeline
from PIL import Image
import matplotlib.pyplot as plt

# Load pipeline with fine-tuned LoRA
print("Loading model...")
pipe = FluxFillPipeline.from_pretrained(
    "xiaozaa/flux1-fill-dev-diffusers",
    torch_dtype=torch.bfloat16
).to("cuda")

# Load LoRA weights
lora_files = glob.glob(f"{OUTPUT_DIR}/**/*.safetensors", recursive=True)
if lora_files:
    lora_path = sorted(lora_files)[-1]
    pipe.load_lora_weights(lora_path)
    print(f"✅ Loaded LoRA: {lora_path}")

# Test with first training sample
pairs_file = f"{DATA_DIR}/train_pairs.txt"
if os.path.exists(pairs_file):
    with open(pairs_file) as f:
        test_pair = f.readline().strip().split()
    
    person = Image.open(f"{DATA_DIR}/train/image/{test_pair[0]}").convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    garment = Image.open(f"{DATA_DIR}/train/cloth/{test_pair[1]}").convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    mask = Image.open(f"{DATA_DIR}/train/agnostic-mask/{test_pair[0].replace('.jpg', '_mask.png')}").convert('L').resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    gt = Image.open(f"{DATA_DIR}/train/result/{test_pair[0]}").convert('RGB').resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    
    # Create inpainting input
    person_arr = np.array(person)
    mask_arr = np.array(mask)
    masked = person_arr.copy()
    masked[mask_arr > 128] = 128  # Gray out masked region
    
    # Concatenate cloth + masked person (CatVTON format)
    concat_input = Image.new('RGB', (IMAGE_WIDTH * 2, IMAGE_HEIGHT))
    concat_input.paste(garment, (0, 0))
    concat_input.paste(Image.fromarray(masked), (IMAGE_WIDTH, 0))
    
    # Generate
    print("Generating...")
    result = pipe(
        image=concat_input,
        mask_image=mask,
        prompt="virtual try-on, photorealistic, high quality",
        height=IMAGE_HEIGHT,
        width=IMAGE_WIDTH,
        num_inference_steps=30,
        guidance_scale=3.5,
    ).images[0]
    
    # Display comparison
    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    axes[0].imshow(person); axes[0].set_title('Person')
    axes[1].imshow(garment); axes[1].set_title('Garment')
    axes[2].imshow(gt); axes[2].set_title('Gemini GT')
    axes[3].imshow(result); axes[3].set_title('Fine-tuned')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("✅ Inference test complete")